<a href="https://colab.research.google.com/github/MohammedAl-Shareef/Quantum-ML-KEM-ML-DSA-SLH-DSA/blob/mlkem-alshareef/ml_kem_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Parameters
n = 4   # Degree of polynomials
q = 17  # Prime modulus for all operations
k = 2   # Matrix dimension (2x2)

# Basic Polynomial Operations

def poly_add(a, b):
    """Add two polynomials coefficient-wise modulo q."""
    return [(ai + bi) % q for ai, bi in zip(a, b)]

def poly_mul(a, b):
    """Multiply two polynomials modulo (X^n + 1) and q."""
    result = [0] * n
    for i in range(n):
        for j in range(n):
            result[(i + j) % n] = (result[(i + j) % n] + a[i] * b[j]) % q
    return result

# Key Generation

def keygen():
    # Example fixed matrix A (for simplicity in testing)
    A = [
        [[1, 0, 0, 0], [2, 1, 3, 0]],
        [[0, 1, 2, 0], [1, 1, 1, 1]],
    ]

    # Secret key s and small error term e
    s = [[1, -1, 0, 1], [0, 1, -1, 0]]
    e = [[1, 0, -1, 1], [0, 1, 0, -1]]

    # Compute public key: t = A·s + e
    t = []
    for i in range(k):
        ti = [0] * n
        for j in range(k):
            ti = poly_add(ti, poly_mul(A[i][j], s[j]))
        ti = poly_add(ti, e[i])
        t.append(ti)

    return (A, t), s

# Encapsulation

def encapsulate(pk):
    A, t = pk
    # Random small polynomials r and errors e1
    r = [[1, -1, 0, 0], [0, 1, -1, 0]]
    e1 = [[0, 1, 0, -1], [1, 0, 0, 0]]
    m = [1, 0, 1, 0]  # Small binary message

    # Compute u = Aᵀ·r + e1
    u = []
    for i in range(k):
        ui = [0] * n
        for j in range(k):
            ui = poly_add(ui, poly_mul(A[j][i], r[j]))  # Transpose access
        ui = poly_add(ui, e1[i])
        u.append(ui)

    # Compute v = tᵀ·r + e2 + encode(m)
    v = [0] * n
    for j in range(k):
        v = poly_add(v, poly_mul(t[j], r[j]))

    e2 = [0, -1, 1, 0]  # Another small error
    v = poly_add(v, e2)

    # Message encoding: 1 → q/2, 0 → 0
    m_encoded = [(q // 2) * bit for bit in m]
    v = poly_add(v, m_encoded)

    return u, v, m

# Decapsulation (corrected version)

def decapsulate(u, v, s):
    # Recover w = v - sᵀ·u
    w = v.copy()
    for i in range(k):
        correction = poly_mul(s[i], u[i])
        w = [(wi - ci) % q for wi, ci in zip(w, correction)]

    # Decode message: closer to q/2 => 1, else 0
    m_decoded = []
    for wi in w:
        if abs(wi - q//2) < abs(wi - 0):
            m_decoded.append(1)
        else:
            m_decoded.append(0)
    return m_decoded

# Main Test

pk, sk = keygen()
u, v, m_original = encapsulate(pk)
m_recovered = decapsulate(u, v, sk)

print("Original Message: ", m_original)
print("Recovered Message:", m_recovered)
print("Success:", m_original == m_recovered)

Original Message:  [1, 0, 1, 0]
Recovered Message: [1, 1, 1, 0]
Success: False
